# 全量数据 + MERT 音乐标签微调
研究用途。先看同目录 README.md。下载阶段用 CPU；训练阶段再切 GPU。Drive 5 TB 是存储，不是显存。这个 notebook 不训练音源分离模型。默认先运行小样本冒烟检查，不会自动启动全量长训练。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib, subprocess, sys, os
BASE = pathlib.Path('/content/drive/MyDrive/Audio AI/MusicMixer')
SRC = BASE / 'project'
DATA = BASE / 'data' / 'jamendo'
BASE.mkdir(parents=True, exist_ok=True)
# 首次：把本次更新后的项目上传/解压到 SRC；也可在更新已推送后 clone。
if not SRC.exists():
    subprocess.run(['git', 'clone', 'https://github.com/EthanBAI-dev/musicmix.git', str(SRC)], check=True)
assert (SRC/'scripts/cloud_train.py').exists(), '云盘项目仍是旧版本，请上传本次文件；不要继续运行旧 notebook'
os.chdir(SRC)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'colab/requirements-train.txt'], check=True)
os.environ['HF_HOME'] = str(BASE/'huggingface')
print('project:', SRC, '\ndata:', DATA)


## 全量下载（00–99，共 100 块）
先确认 Drive 实际剩余空间至少约 600 GB，额外为模型/训练留余量。下面默认不自动下载；设为 True 执行。原始 tar 永久保留在 Drive，断线后重跑续传，不铺开五万多个音频文件。不是放进高 RAM。

In [ ]:
DOWNLOAD_ALL = False
if DOWNLOAD_ALL:
    subprocess.run([sys.executable, '-m', 'scripts.cloud_download', '--root', str(DATA), '--start', '0', '--stop', '100'], check=True)


## 固定模型版本与训练设置
MODE='frozen' 是对照；'last' 微调最后两层；'full' 微调所有参数。默认 95M/10 秒/batch 2/累积 8，不保证所有 GPU 均可运行。OOM 时先减 batch，再减片段长度；改变配置用新 RUN_NAME。模型加载会执行官方远程代码，版本 SHA 首次解析后持久保存，应审核该版本。

In [ ]:
import torch, json
from huggingface_hub import model_info
assert torch.cuda.is_available(), '切换为 GPU 运行时'
print(torch.cuda.get_device_name(0), 'VRAM GB:', torch.cuda.get_device_properties(0).total_memory/2**30)
MODEL = 'm-a-p/MERT-v1-95M'
lockfile = BASE/(MODEL.split('/')[-1] + '-revision.json')
if not lockfile.exists():
    lockfile.write_text(json.dumps({'model': MODEL, 'revision': model_info(MODEL).sha}))
REVISION = json.loads(lockfile.read_text())['revision']
MODE = 'last'
SMOKE = True
RESUME = False
RUN_NAME = ('smoke-' if SMOKE else 'full-data-') + MODE + '-seed0'
OUT = BASE/'runs'/RUN_NAME
CMD = [sys.executable, '-m', 'scripts.cloud_train', '--root', str(DATA), '--out', str(OUT), '--model', MODEL, '--revision', REVISION, '--mode', MODE, '--epochs', '1' if SMOKE else '10', '--batch-size', '2', '--accumulation', '8', '--seconds', '10', '--eval-segments', '3', '--seed', '0']
if SMOKE: CMD += ['--smoke-limit', '64']
print('revision:', REVISION, '\nrun:', OUT)


In [ ]:
subprocess.run(CMD + (['--resume'] if RESUME else []), check=True)


## 最后才评测试集
先在验证集完成 frozen/last/不同种子的选择，再运行一次最终测试。旧测试集已经多次参与实验讨论；不能把本次结果包装成完全未接触过的盲测。正式求职报告还应增加独立数据或预注册的留出评估。

In [ ]:
FINAL_TEST = False
if FINAL_TEST:
    assert not SMOKE, '冒烟结果不能作为正式结果'
    subprocess.run(CMD + ['--evaluate-test'], check=True)
